[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/robomotic/mjlab/blob/feature/motor-database-extension/notebooks/electrical/01_intro.ipynb)

# Tutorial 1: Electric Motor Fundamentals

## 🎯 Learning Objectives

By the end of this tutorial, you will understand:
- **Motor types** used in robotics: BLDC, PMSM, brushed vs brushless servos
- **Electrical physics equations** that govern motor behavior
- **Power consumption characteristics** through isolated motor bench testing
- **How torque and speed affect** current, voltage, and temperature

**Prerequisites:** Basic physics (voltage, current, resistance)

**Time:** ~45 minutes

## 📦 Setup (Colab Only)

This cell installs mjlab if you're running in Google Colab.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
  print("🔧 Running in Google Colab - setting up environment...\n")

  # Install mjlab
  print("📦 Installing mjlab...")
  !pip install --upgrade moviepy
  # Step 1: install all dependencies (uses pip cache, skips what's already installed)
  !pip install -q git+https://github.com/robomotic/mjlab.git@feature/motor-database-extension
  # Step 2: force-reinstall only mjlab so the latest branch commit is always used,
  # without re-downloading heavy deps (mujoco, torch, warp, tyro, etc.)
  !pip install -q --force-reinstall --no-deps git+https://github.com/robomotic/mjlab.git@feature/motor-database-extension

  print("\n✅ Setup complete! Ready to run the tutorial.")
else:
  print("💻 Running locally - assuming mjlab is already installed")

## 🔌 Motor Types in Robotics

Modern robots use various types of electric motors. Let's understand the key differences:

### 1. Brushed DC Motors

**How they work:**
- Use mechanical **brushes** (carbon contacts) to commutate (switch) current
- Simple control: voltage in → torque out
- Rotor has windings, stator has permanent magnets

**Advantages:**
- ✅ Simple control (just apply voltage)
- ✅ Low cost
- ✅ Good low-speed torque

**Disadvantages:**
- ❌ Brushes wear out (limited lifetime ~2000 hours)
- ❌ Lower efficiency (~75-80%)
- ❌ Electromagnetic noise from sparking
- ❌ Requires regular maintenance

**Usage in robotics:** Older hobby servos, simple actuators

---

### 2. Brushless DC Motors (BLDC)

**How they work:**
- Use **electronic commutation** (no brushes!)
- Rotor has permanent magnets, stator has windings
- Requires ESC (Electronic Speed Controller) to switch currents
- Typically uses trapezoidal commutation (6-step)

**Advantages:**
- ✅ High efficiency (~85-90%)
- ✅ Long lifetime (no brush wear)
- ✅ High power density (compact)
- ✅ Low maintenance

**Disadvantages:**
- ❌ Requires electronic controller (ESC)
- ❌ More complex control
- ❌ Higher cost

**Usage in robotics:** Drones (quadcopter motors), small actuators

---

### 3. Permanent Magnet Synchronous Motors (PMSM)

**How they work:**
- Similar to BLDC but uses **sinusoidal commutation** (FOC - Field Oriented Control)
- Smoother torque output (no cogging)
- Rotor has permanent magnets, stator has 3-phase windings
- Requires advanced controller with position feedback

**Advantages:**
- ✅ Highest efficiency (~90-95%)
- ✅ Smooth torque (no ripple)
- ✅ Excellent controllability
- ✅ High power density
- ✅ Quiet operation

**Disadvantages:**
- ❌ Complex control (FOC requires significant computation)
- ❌ Higher cost
- ❌ Requires high-resolution encoder

**Usage in robotics:** **Most modern humanoid/quadruped robots**
- Unitree G1/H1/Go2 actuators
- Boston Dynamics actuators
- Tesla Optimus actuators

---

### Comparison Table

| Feature | Brushed DC | BLDC | PMSM |
|---------|------------|------|------|
| **Commutation** | Mechanical | Electronic (trapezoidal) | Electronic (sinusoidal/FOC) |
| **Efficiency** | 75-80% | 85-90% | 90-95% |
| **Lifetime** | ~2000 hrs | >20,000 hrs | >20,000 hrs |
| **Torque Ripple** | Moderate | Low | Very low |
| **Control Complexity** | Simple | Moderate | High |
| **Cost** | $ | $$ | $$$ |
| **Robotics Usage** | Hobby servos | Drones, small actuators | Humanoids, quadrupeds |

**mjlab focuses on PMSM/BLDC** since these are used in modern research robots (Unitree, ANYmal, Spot, etc.).

## ⚡ Electrical Physics Equations

mjlab's `ElectricalMotorActuator` implements the following physics equations to model realistic motor behavior.

### 1. Voltage Equation (RL Circuit)

The motor acts as an **RL circuit** with back-EMF:

$$
V = I \cdot R + L \cdot \frac{dI}{dt} + K_e \cdot \omega
$$

Where:
- $V$ = applied voltage (volts)
- $I$ = motor current (amperes)
- $R$ = winding resistance (ohms) - typically 0.1-1.0 Ω
- $L$ = winding inductance (henries) - typically 0.0001-0.001 H
- $K_e$ = back-EMF constant (V/(rad/s)) - typically 0.05-0.2
- $\omega$ = motor angular velocity (rad/s)

**Physical meaning:**
- $I \cdot R$ : Resistive voltage drop (power loss as heat)
- $L \cdot \frac{dI}{dt}$ : Inductive voltage drop (current can't change instantly)
- $K_e \cdot \omega$ : **Back-EMF** (motor generates voltage opposing applied voltage when spinning)

**Key insight:** At high speeds, back-EMF reduces available voltage for current, limiting torque!

---

### 2. Torque Equation

$$
\tau = K_t \cdot I
$$

Where:
- $\tau$ = motor torque (N⋅m)
- $K_t$ = torque constant (N⋅m/A) - typically 0.05-0.2
- $I$ = motor current (amperes)

**Key insight:** More current = more torque (linear relationship)

**Note:** For PMSM motors in SI units, $K_t = K_e$ (same numerical value)

---

### 3. Mechanical Power Equation

$$
P_{mech} = \tau \cdot \omega
$$

Where:
- $P_{mech}$ = mechanical power output (watts)
- $\tau$ = motor torque (N⋅m)
- $\omega$ = angular velocity (rad/s)

**Key insight:** Power = torque × speed. High-speed or high-torque both require high power.

---

### 4. Electrical Power Equation

$$
P_{elec} = V \cdot I
$$

Where:
- $P_{elec}$ = electrical power input (watts)
- $V$ = applied voltage (volts)
- $I$ = motor current (amperes)

---

### 5. Power Loss (Heat Generation)

$$
P_{loss} = I^2 \cdot R
$$

Where:
- $P_{loss}$ = power dissipated as heat (watts)
- $I$ = motor current (amperes)
- $R$ = winding resistance (ohms)

**Key insight:** Heat increases with **square of current**! Double current = 4× heat.

---

### 6. Motor Efficiency

$$
\eta = \frac{P_{mech}}{P_{elec}} = \frac{\tau \cdot \omega}{V \cdot I}
$$

Typical values:
- PMSM: 90-95%
- BLDC: 85-90%
- Brushed DC: 75-80%

---

### 7. Thermal Dynamics

$$
\frac{dT}{dt} = \frac{I^2 \cdot R - \frac{T - T_{amb}}{R_{th}}}{\tau_{th}}
$$

Where:
- $T$ = winding temperature (°C)
- $T_{amb}$ = ambient temperature (°C) - typically 25°C
- $R_{th}$ = thermal resistance (°C/W) - typically 3-10 °C/W
- $\tau_{th}$ = thermal time constant (seconds) - typically 60-300s

**Physical meaning:**
- $I^2 \cdot R$ : Heat generation rate (watts)
- $\frac{T - T_{amb}}{R_{th}}$ : Heat dissipation rate (watts)
- $\tau_{th}$ : How fast temperature changes (thermal mass)

**Key insight:** Temperature rises until heat generation = heat dissipation (steady state)

---

### Summary: Energy Flow

```
Battery          Motor Windings       Mechanical Output
  |
  V
P_elec = V·I  →  [I²R loss → Heat]  →  P_mech = τ·ω
                        ↓
                  Temperature ↑
```

**Conservation of energy:**
$$P_{elec} = P_{mech} + P_{loss}$$

mjlab automatically computes all these quantities at each simulation step!

## 🔬 Bench Testing: Isolated Motor Experiments

Let's test a single motor on a "virtual benchtop" to understand how torque and speed affect electrical properties.

We'll use the **Dynamixel XL330-M288-T** - a small brushed DC servo motor commonly used in robotics education and hobby projects. This motor has complete specifications, making it perfect for learning!

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from mjlab.motor_database import load_motor_spec

print("✅ Imports successful!")

### Load Motor Specification

mjlab includes a database of real motor specifications, and can automatically fetch community-maintained specs from GitHub!

For this tutorial, we'll use the **Faulhaber 2264W024BP4** - a high-quality brushless DC motor from Faulhaber's extensive catalog. This motor is commonly used in precision robotics and medical devices.

**From datasheet (at 24V):**
- Stall torque: 1.311 N⋅m, Stall current: 109 A
- No-load speed: 2209.6 rad/s (21,100 RPM)
- This is a direct-drive motor (gear_ratio = 1.0)

**Calculated from physics:**
- **Torque constant**: Kt = τ / I = 1.311 / 109 = **0.0118 N⋅m/A** ✓
- **Resistance**: R = V / I_stall = 24.0 / 109 = **0.22 Ω** ✓  
- **Back-EMF constant**: Ke = Kt = **0.0118 V/(rad/s)** ✓

This motor will be **automatically downloaded from GitHub** on first use and cached locally!

In [ ]:
# Load Faulhaber 2264W024BP4 motor (automatically fetches from GitHub)
motor = load_motor_spec("faulhaber_2264w024bp4")

print("🔧 Motor Specification: Faulhaber 2264W024BP4\n")
print("Type: Brushless DC Motor (BLDC) - Direct Drive\n")
print("Mechanical Properties:")
print(f"  Gear ratio:        {motor.gear_ratio}:1 (direct drive)")
print(f"  Stall torque:      {motor.stall_torque} N⋅m")
print(f"  Continuous torque: {motor.continuous_torque} N⋅m")
print(f"  Peak torque:       {motor.peak_torque} N⋅m")
print(
  f"  No-load speed:     {motor.no_load_speed:.2f} rad/s ({motor.no_load_speed * 9.549:.0f} RPM)"
)
print("\nElectrical Properties:")
print(f"  Voltage range:     {motor.voltage_range[0]}-{motor.voltage_range[1]} V")
print(f"  Resistance (R):    {motor.resistance} Ω")
print(f"  Inductance (L):    {motor.inductance * 1000:.2f} mH")
print(f"  Kt (torque const): {motor.motor_constant_kt} N⋅m/A")
print(f"  Ke (back-EMF):     {motor.motor_constant_ke} V/(rad/s)")
print(f"  Stall current:     {motor.stall_current} A")
print(f"  No-load current:   {motor.no_load_current * 1000:.0f} mA")
print("\nThermal Properties:")
print(f"  Thermal resistance: {motor.thermal_resistance} °C/W")
print(f"  Time constant:      {motor.thermal_time_constant:.0f} seconds")
print(f"  Max temperature:    {motor.max_winding_temperature} °C")
print(f"  Ambient temp:       {motor.ambient_temperature} °C")
print("\n✨ Motor spec automatically downloaded from GitHub!")
print("   Source: https://github.com/robomotic/mujoco-motors")
print("   Cached at: ~/.mjlab/cache/motors/")

### Experiment 1: Current vs Torque (Motor Constant Kt)

According to the equation $\tau = K_t \cdot I$, torque should be linear with current.

Let's verify this by computing current for different torque demands:

In [ ]:
# Torque range: 0 to peak torque
torques = np.linspace(0, motor.peak_torque, 50)

# Calculate required current: I = τ / Kt
currents = torques / motor.motor_constant_kt

# Calculate power loss (heat): P_loss = I²R
power_losses = currents**2 * motor.resistance

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
  "Experiment 1: Torque-Current Relationship", fontsize=14, fontweight="bold"
)

# Current vs Torque
ax1.plot(torques, currents, "b-", linewidth=2.5, label="Required current")
ax1.axhline(
  y=motor.stall_current,
  color="r",
  linestyle="--",
  alpha=0.7,
  label=f"Stall current ({motor.stall_current}A)",
)
ax1.axhline(
  y=motor.operating_current,
  color="orange",
  linestyle="--",
  alpha=0.7,
  label=f"Operating current ({motor.operating_current}A)",
)
ax1.axvline(
  x=motor.continuous_torque,
  color="g",
  linestyle="--",
  alpha=0.7,
  label=f"Continuous torque ({motor.continuous_torque}N⋅m)",
)
ax1.set_xlabel("Torque (N⋅m)", fontsize=11)
ax1.set_ylabel("Current (A)", fontsize=11)
ax1.set_title("Linear Relationship: I = τ / Kt", fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.legend()

# Power loss vs Torque
ax2.plot(torques, power_losses, "r-", linewidth=2.5, label="I²R losses (heat)")
ax2.set_xlabel("Torque (N⋅m)", fontsize=11)
ax2.set_ylabel("Power Loss (W)", fontsize=11)
ax2.set_title("Heat Generation (Quadratic with Current)", fontsize=12)
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

print("\n📊 Key Observations:")
print(
  f"   At continuous torque ({motor.continuous_torque}N⋅m): {motor.continuous_torque / motor.motor_constant_kt:.1f}A current"
)
print(
  f"   At peak torque ({motor.peak_torque}N⋅m): {motor.peak_torque / motor.motor_constant_kt:.1f}A current"
)
print(
  f"   Heat at continuous: {(motor.continuous_torque / motor.motor_constant_kt) ** 2 * motor.resistance:.1f}W"
)
print(
  f"   Heat at peak: {(motor.peak_torque / motor.motor_constant_kt) ** 2 * motor.resistance:.1f}W"
)
print("\n⚠️  Notice: Heat grows as I² - doubling torque quadruples heat!")

### Experiment 2: Back-EMF and Speed Limits

The back-EMF voltage $V_{emf} = K_e \cdot \omega$ opposes the applied voltage.

At high speeds, most of the voltage goes to back-EMF, leaving little for current (and thus torque).

Let's see how available torque decreases with speed:

In [ ]:
# Speed range: 0 to no-load speed
speeds = np.linspace(0, motor.no_load_speed, 100)

# Applied voltage (rated voltage)
V_applied = motor.voltage_range[1]  # 24V

# Back-EMF: V_emf = Ke * ω
V_emf = motor.motor_constant_ke * speeds

# Available voltage for current: V_available = V_applied - V_emf - I*R
# At stall torque: I = τ_stall / Kt
I_stall = motor.stall_torque / motor.motor_constant_kt
V_resistive_stall = I_stall * motor.resistance

# Maximum current at each speed (ignoring resistance for simplicity)
I_max = (V_applied - V_emf) / motor.resistance
I_max = np.maximum(I_max, 0)  # Can't be negative

# Maximum torque at each speed: τ = Kt * I
torque_max = motor.motor_constant_kt * I_max

# Mechanical power: P = τ * ω
power_mech = torque_max * speeds

# Plot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
  "Experiment 2: Back-EMF and Speed-Torque Characteristics",
  fontsize=14,
  fontweight="bold",
)

# Back-EMF vs Speed
axes[0, 0].plot(speeds, V_emf, "b-", linewidth=2.5, label="Back-EMF")
axes[0, 0].axhline(
  y=V_applied,
  color="r",
  linestyle="--",
  alpha=0.7,
  label=f"Applied voltage ({V_applied}V)",
)
axes[0, 0].set_xlabel("Speed (rad/s)", fontsize=11)
axes[0, 0].set_ylabel("Voltage (V)", fontsize=11)
axes[0, 0].set_title("Back-EMF Increases with Speed", fontsize=12)
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

# Available current vs Speed
axes[0, 1].plot(speeds, I_max, "g-", linewidth=2.5, label="Max current")
axes[0, 1].axhline(
  y=motor.stall_current,
  color="r",
  linestyle="--",
  alpha=0.7,
  label=f"Stall current ({motor.stall_current}A)",
)
axes[0, 1].set_xlabel("Speed (rad/s)", fontsize=11)
axes[0, 1].set_ylabel("Current (A)", fontsize=11)
axes[0, 1].set_title("Available Current Decreases with Speed", fontsize=12)
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

# Torque-speed curve
axes[1, 0].plot(speeds, torque_max, "purple", linewidth=2.5, label="Max torque")
axes[1, 0].axhline(
  y=motor.continuous_torque,
  color="orange",
  linestyle="--",
  alpha=0.7,
  label=f"Continuous ({motor.continuous_torque}N⋅m)",
)
axes[1, 0].axhline(
  y=motor.stall_torque,
  color="r",
  linestyle="--",
  alpha=0.7,
  label=f"Stall ({motor.stall_torque}N⋅m)",
)
axes[1, 0].set_xlabel("Speed (rad/s)", fontsize=11)
axes[1, 0].set_ylabel("Torque (N⋅m)", fontsize=11)
axes[1, 0].set_title("Speed-Torque Curve (Linear Drop)", fontsize=12)
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

# Mechanical power vs Speed
axes[1, 1].plot(speeds, power_mech, "orange", linewidth=2.5, label="Mechanical power")
max_power_idx = np.argmax(power_mech)
axes[1, 1].plot(
  speeds[max_power_idx],
  power_mech[max_power_idx],
  "ro",
  markersize=10,
  label=f"Peak power ({power_mech[max_power_idx]:.0f}W)",
)
axes[1, 1].set_xlabel("Speed (rad/s)", fontsize=11)
axes[1, 1].set_ylabel("Power (W)", fontsize=11)
axes[1, 1].set_title("Power Curve (Peaks at Mid-Speed)", fontsize=12)
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print("\n📊 Key Observations:")
print(f"   At zero speed: {torque_max[0]:.1f} N⋅m torque available (stall torque)")
print(
  f"   At no-load speed ({motor.no_load_speed:.1f} rad/s): {torque_max[-1]:.1f} N⋅m torque (≈0)"
)
print(
  f"   Peak power: {power_mech[max_power_idx]:.0f}W at {speeds[max_power_idx]:.1f} rad/s"
)
print("\n⚠️  Trade-off: Can't have both high torque AND high speed simultaneously!")

### Experiment 3: Thermal Behavior Over Time

Let's simulate continuous operation at different torque levels and see temperature rise.

In [ ]:
# Simulate thermal dynamics for 10 minutes at different torques
time_seconds = 600  # 10 minutes
dt = 1.0  # 1 second timestep
steps = int(time_seconds / dt)

# Test three torque levels (IMPORTANT: only continuous torque is safe!)
test_torques = [
  motor.continuous_torque * 0.5,  # 50% continuous - SAFE
  motor.continuous_torque,  # 100% continuous - SAFE
  motor.continuous_torque * 1.5,  # 150% continuous - MARGINAL
]

test_labels = [
  f"50% continuous ({test_torques[0]:.3f}N⋅m)",
  f"100% continuous ({test_torques[1]:.3f}N⋅m)",
  f"150% continuous ({test_torques[2]:.3f}N⋅m)",
]
test_colors = ["green", "orange", "red"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
  "Experiment 3: Thermal Behavior Under Constant Load", fontsize=14, fontweight="bold"
)

for torque, label, color in zip(test_torques, test_labels, test_colors, strict=False):
  # Calculate current and power loss
  current = torque / motor.motor_constant_kt
  power_loss = current**2 * motor.resistance

  # Simulate temperature over time
  temperatures = [motor.ambient_temperature]
  times = [0]

  T = motor.ambient_temperature
  for step in range(steps):
    # dT/dt = (I²R - (T-T_amb)/R_th) / τ_th
    heat_in = power_loss
    heat_out = (T - motor.ambient_temperature) / motor.thermal_resistance
    dT_dt = (heat_in - heat_out) / motor.thermal_time_constant

    T += dT_dt * dt
    temperatures.append(T)
    times.append((step + 1) * dt)

  # Plot temperature
  ax1.plot(
    np.array(times) / 60,
    temperatures,
    color=color,
    linewidth=2.5,
    label=label,
  )

  # Plot steady-state analysis
  steady_state_temp = motor.ambient_temperature + power_loss * motor.thermal_resistance
  ax1.axhline(y=steady_state_temp, color=color, linestyle="--", alpha=0.3)

# Temperature limits
ax1.axhline(
  y=motor.max_winding_temperature,
  color="r",
  linestyle="--",
  linewidth=2,
  alpha=0.7,
  label=f"Max temp ({motor.max_winding_temperature}°C)",
)
ax1.set_xlabel("Time (minutes)", fontsize=11)
ax1.set_ylabel("Temperature (°C)", fontsize=11)
ax1.set_title("Temperature Rise Over Time", fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.legend()

# Steady-state temperature vs torque (FIXED: limit to reasonable range!)
# Only plot up to 3x continuous torque (beyond that, motor fails before steady-state)
max_safe_torque = motor.continuous_torque * 3.0
torque_range = np.linspace(0, max_safe_torque, 50)
currents_range = torque_range / motor.motor_constant_kt
power_loss_range = currents_range**2 * motor.resistance
steady_temps = motor.ambient_temperature + power_loss_range * motor.thermal_resistance

ax2.plot(torque_range, steady_temps, "b-", linewidth=2.5, label="Steady-state temp")
ax2.axhline(
  y=motor.max_winding_temperature,
  color="r",
  linestyle="--",
  linewidth=2,
  alpha=0.7,
  label=f"Max temp ({motor.max_winding_temperature}°C)",
)
ax2.axvline(
  x=motor.continuous_torque,
  color="g",
  linestyle="--",
  alpha=0.7,
  label=f"Continuous torque ({motor.continuous_torque:.3f}N⋅m)",
)
ax2.axvline(
  x=motor.peak_torque,
  color="r",
  linestyle="--",
  alpha=0.7,
  label=f"Peak torque ({motor.peak_torque:.2f}N⋅m) - pulses only!",
)

# Add danger zone
ax2.axvspan(motor.continuous_torque * 2, max_safe_torque, alpha=0.2, color="red")
ax2.text(
  motor.continuous_torque * 2.3,
  motor.max_winding_temperature * 0.8,
  "DANGER ZONE\n(Motor fails in seconds)",
  fontsize=10,
  color="red",
  fontweight="bold",
)

ax2.set_xlabel("Torque (N⋅m)", fontsize=11)
ax2.set_ylabel("Steady-State Temperature (°C)", fontsize=11)
ax2.set_title("Thermal Limit vs Torque (Limited to Safe Range)", fontsize=12)
ax2.set_xlim(0, 0.2)  # Focus on beginning where temperature rises steeply
ax2.set_ylim(0, motor.max_winding_temperature * 2)  # Limit Y-axis for readability
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

print("\n📊 Key Observations:")
print(
  f"   50% load: Stabilizes at {motor.ambient_temperature + (test_torques[0] / motor.motor_constant_kt) ** 2 * motor.resistance * motor.thermal_resistance:.1f}°C ✓ Safe"
)
print(
  f"   100% continuous load: Stabilizes at {motor.ambient_temperature + (motor.continuous_torque / motor.motor_constant_kt) ** 2 * motor.resistance * motor.thermal_resistance:.1f}°C ✓ Safe"
)
print(
  f"   150% load: Stabilizes at {motor.ambient_temperature + (test_torques[2] / motor.motor_constant_kt) ** 2 * motor.resistance * motor.thermal_resistance:.1f}°C ⚠️ Marginal"
)

print(
  f"\n⚠️ WARNING: Continuous torque ({motor.continuous_torque:.3f}N⋅m) << Peak torque ({motor.peak_torque:.2f}N⋅m)"
)
print(f"   Ratio: {motor.peak_torque / motor.continuous_torque:.0f}:1")
print(
  "   Peak torque is only for SHORT PULSES (milliseconds), not continuous operation!"
)

print(f"\n🔥 At peak torque ({motor.peak_torque}N⋅m):")
peak_current = motor.peak_torque / motor.motor_constant_kt
peak_power_loss = peak_current**2 * motor.resistance
peak_temp_theoretical = (
  motor.ambient_temperature + peak_power_loss * motor.thermal_resistance
)
print(f"   Current: {peak_current:.0f}A")
print(f"   Heat generation: {peak_power_loss:.0f}W")
print(f"   Theoretical steady-state temp: {peak_temp_theoretical:.0f}°C")
print("   ❌ Motor would FAIL in seconds - never reaches steady-state!")
print("   ❌ This is why peak torque is rated for pulses only!")

### 🔍 Why This Matters: Key Insights

#### 1. **No-Load Speed ≠ Zero Power**

Even at maximum speed (no external load), the motor consumes **6.3W**:
- Current: 0.261 A (very small, but not zero!)
- Power goes to: internal friction, bearing losses, windage
- Efficiency: **0%** because no useful work is done (τ = 0)

**Practical Impact:** Running motors at high speed with no load drains battery slowly but constantly.

---

#### 2. **Stall Condition = Maximum Power Loss**

At stall (blocked shaft), the motor consumes **2,616W** as pure heat:
- Current: 109 A (maximum!)
- All electrical power → heat (no mechanical work since ω = 0)
- Efficiency: **0%** because motor isn't moving
- **DANGER:** Motor will overheat and fail within seconds!

**Practical Impact:** Never block a powered motor - it will burn out from thermal overload.

---

#### 3. **Half-Torque = Peak Efficiency**

The sweet spot is around 50% torque/speed:
- Good mechanical power output: 1,445W
- Acceptable efficiency: ~52%
- Moderate heat generation: 1,063W
- Motor can sustain this longer than stall or high-torque conditions

**Practical Impact:** For maximum efficiency and battery life, operate motors in the mid-range of their torque-speed curve.

---

#### 4. **The Power Budget Reality**

| Condition | Electrical Power In | Useful Mechanical Work | Wasted as Heat | Efficiency |
|-----------|---------------------|------------------------|----------------|------------|
| No-load   | 6.3 W               | 0 W (0%)               | 6.3 W          | 0% |
| Half-torque | 2,508 W           | 1,445 W (58%)          | 1,063 W        | 52% |
| Stall     | 2,616 W             | 0 W (0%)               | 2,616 W        | 0% |

**Key Takeaway:** Motors are only efficient when doing actual work (both torque AND speed). The extremes (no-load, stall) have 0% efficiency!

---

#### 5. **Battery Life Implications**

For a 24V, 10Ah battery (240 Wh capacity):

| Condition | Power Draw | Battery Life |
|-----------|------------|--------------|
| No-load   | 6.3 W      | **38 hours** ⚡ |
| Half-torque | 2,508 W  | **5.7 minutes** 🔋 |
| Stall     | 2,616 W    | **5.5 minutes** 🔥 |

**Critical Insight:** A robot idling (motors at no-load) drains battery 400× slower than under load!

---

#### 6. **Thermal Management**

Heat generation follows current squared (I²R):

- No-load: 0.015 W (negligible - can run forever)
- Half-torque: 1,063 W (manageable for continuous operation)
- Stall: 2,616 W (thermal shutdown in ~30 seconds!)

**Design Rule:** Always size motors so continuous operation stays well below stall conditions.

---

### 💡 Practical Design Guidelines

1. **Avoid sustained high-torque operation** - kills battery and causes overheating
2. **No-load spinning is OK** - minimal power consumption, but no useful work
3. **Design for mid-range efficiency** - size motors so typical loads are 30-60% of stall torque
4. **Add current limiting** - prevent accidental stall conditions from burning out motors
5. **Monitor temperature** - thermal limits protect against sustained overload

**This is why professional robot actuators include:**
- Current sensors (detect overload)
- Temperature sensors (thermal protection)
- Torque limits in firmware (prevent stall)
- Battery management systems (prevent over-discharge)

In [ ]:
# Three key operating conditions
conditions = ["No-Load\n(Max Speed)", "Half Torque\n(Mid Speed)", "Stall\n(Max Torque)"]

# Condition 1: No-load speed (zero torque, maximum speed)
torque_noload = 0.0
speed_noload = motor.no_load_speed
current_noload = motor.no_load_current
voltage_noload = motor.voltage_range[1]  # Full voltage
power_elec_noload = voltage_noload * current_noload
power_mech_noload = torque_noload * speed_noload  # Zero mechanical power!
power_loss_noload = current_noload**2 * motor.resistance
efficiency_noload = 0.0  # Zero mechanical power out

# Condition 2: Half torque at corresponding speed
torque_half = motor.stall_torque / 2
current_half = torque_half / motor.motor_constant_kt
# At half torque, speed is reduced due to back-EMF
# From V = I*R + Ke*ω, solving for ω:
speed_half = (
  voltage_noload - current_half * motor.resistance
) / motor.motor_constant_ke
power_elec_half = voltage_noload * current_half
power_mech_half = torque_half * speed_half
power_loss_half = current_half**2 * motor.resistance
efficiency_half = (
  (power_mech_half / power_elec_half) * 100 if power_elec_half > 0 else 0
)

# Condition 3: Stall (maximum torque, zero speed)
torque_stall = motor.stall_torque
speed_stall = 0.0
current_stall = motor.stall_current
power_elec_stall = voltage_noload * current_stall
power_mech_stall = torque_stall * speed_stall  # Zero mechanical power!
power_loss_stall = current_stall**2 * motor.resistance
efficiency_stall = 0.0  # Zero mechanical power out

# Store data
currents = [current_noload, current_half, current_stall]
speeds = [speed_noload, speed_half, speed_stall]
torques = [torque_noload, torque_half, torque_stall]
power_elec = [power_elec_noload, power_elec_half, power_elec_stall]
power_mech = [power_mech_noload, power_mech_half, power_mech_stall]
power_loss = [power_loss_noload, power_loss_half, power_loss_stall]
efficiencies = [efficiency_noload, efficiency_half, efficiency_stall]

# Plot efficiency comparison
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
fig.suptitle(
  "Power Consumption Analysis: No-Load vs Half-Torque vs Stall",
  fontsize=14,
  fontweight="bold",
)

bars = ax.bar(conditions, efficiencies, color=["green", "orange", "red"], alpha=0.7)
ax.set_ylabel("Efficiency (%)", fontsize=11, fontweight="bold")
ax.set_title("Motor Efficiency (Pmech / Pelec)", fontsize=12)
ax.set_ylim(0, 100)
ax.grid(axis="y", alpha=0.3)
for i, (bar, val) in enumerate(zip(bars, efficiencies)):
  ax.text(
    bar.get_x() + bar.get_width() / 2,
    bar.get_height() + 5,
    f"{val:.1f}%",
    ha="center",
    fontsize=11,
    fontweight="bold",
  )

plt.tight_layout()
plt.show()

# Print detailed comparison
print("\n" + "=" * 80)
print("POWER CONSUMPTION ANALYSIS: THREE KEY OPERATING CONDITIONS")
print("=" * 80)
print(
  f"\n{'Condition':<20} {'No-Load':<15} {'Half-Torque':<15} {'Stall':<15} {'Unit':<10}"
)
print("-" * 80)
print(
  f"{'Current':<20} {current_noload:<15.2f} {current_half:<15.1f} {current_stall:<15.1f} {'A':<10}"
)
print(
  f"{'Speed':<20} {speed_noload:<15.1f} {speed_half:<15.1f} {speed_stall:<15.1f} {'rad/s':<10}"
)
print(
  f"{'Torque':<20} {torque_noload:<15.2f} {torque_half:<15.2f} {torque_stall:<15.2f} {'N⋅m':<10}"
)
print(
  f"{'Elec. Power In':<20} {power_elec_noload:<15.1f} {power_elec_half:<15.1f} {power_elec_stall:<15.1f} {'W':<10}"
)
print(
  f"{'Mech. Power Out':<20} {power_mech_noload:<15.1f} {power_mech_half:<15.1f} {power_mech_stall:<15.1f} {'W':<10}"
)
print(
  f"{'Heat Loss':<20} {power_loss_noload:<15.1f} {power_loss_half:<15.1f} {power_loss_stall:<15.1f} {'W':<10}"
)
print(
  f"{'Efficiency':<20} {efficiency_noload:<15.1f} {efficiency_half:<15.1f} {efficiency_stall:<15.1f} {'%':<10}"
)
print("=" * 80)

### Experiment 4: Power Consumption Across Operating Conditions

A common misconception is that "no-load speed" means the motor consumes no power. Let's examine power consumption at three key operating points:

1. **No-load speed** (maximum speed, zero torque)
2. **Half torque** (50% of stall torque, mid-speed)
3. **Stall** (maximum torque, zero speed)

Understanding this is critical for battery life estimation and thermal management!

In [ ]:
# Load both motors (automatically from GitHub)
motor_large = load_motor_spec("maxon_ec_i_40_488607")  # Larger motor
motor_small = load_motor_spec("faulhaber_2264w024bp4")  # Smaller motor

print("📊 Motor Comparison: Maxon vs Faulhaber\n")
print(f"{'Property':<25} {'Maxon EC-i 40':<20} {'Faulhaber 2264':<20}")
print("=" * 65)
print(
  f"{'Continuous torque':<25} {motor_large.continuous_torque:<20.2f} {motor_small.continuous_torque:<20.2f} N⋅m"
)
print(
  f"{'Peak torque':<25} {motor_large.peak_torque:<20.2f} {motor_small.peak_torque:<20.2f} N⋅m"
)
print(
  f"{'No-load speed':<25} {motor_large.no_load_speed:<20.1f} {motor_small.no_load_speed:<20.1f} rad/s"
)
print(
  f"{'Stall current':<25} {motor_large.stall_current:<20.1f} {motor_small.stall_current:<20.1f} A"
)
print(
  f"{'Resistance':<25} {motor_large.resistance:<20.2f} {motor_small.resistance:<20.2f} Ω"
)
print(
  f"{'Kt (torque constant)':<25} {motor_large.motor_constant_kt:<20.4f} {motor_small.motor_constant_kt:<20.4f} N⋅m/A"
)
print(
  f"{'Thermal resistance':<25} {motor_large.thermal_resistance:<20.1f} {motor_small.thermal_resistance:<20.1f} °C/W"
)

# Compare torque-speed curves
speeds = np.linspace(0, max(motor_large.no_load_speed, motor_small.no_load_speed), 100)

# Large motor
V_large = motor_large.voltage_range[1]
V_emf_large = motor_large.motor_constant_ke * speeds
I_large = np.maximum((V_large - V_emf_large) / motor_large.resistance, 0)
torque_large = motor_large.motor_constant_kt * I_large

# Small motor
V_small = motor_small.voltage_range[1]
V_emf_small = motor_small.motor_constant_ke * speeds
I_small = np.maximum((V_small - V_emf_small) / motor_small.resistance, 0)
torque_small = motor_small.motor_constant_kt * I_small

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
  "Motor Comparison: Maxon EC-i 40 vs Faulhaber 2264", fontsize=14, fontweight="bold"
)

# Torque-speed curves
ax1.plot(speeds, torque_large, "b-", linewidth=2.5, label="Maxon EC-i 40 (2.08 N⋅m)")
ax1.plot(speeds, torque_small, "r-", linewidth=2.5, label="Faulhaber 2264 (1.31 N⋅m)")
ax1.axhline(y=motor_large.continuous_torque, color="b", linestyle="--", alpha=0.3)
ax1.axhline(y=motor_small.continuous_torque, color="r", linestyle="--", alpha=0.3)
ax1.set_xlabel("Speed (rad/s)", fontsize=11)
ax1.set_ylabel("Torque (N⋅m)", fontsize=11)
ax1.set_title("Speed-Torque Curves", fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.legend()

# Power curves
power_large = torque_large * speeds
power_small = torque_small * speeds
ax2.plot(speeds, power_large, "b-", linewidth=2.5, label="Maxon EC-i 40")
ax2.plot(speeds, power_small, "r-", linewidth=2.5, label="Faulhaber 2264")
ax2.set_xlabel("Speed (rad/s)", fontsize=11)
ax2.set_ylabel("Mechanical Power (W)", fontsize=11)
ax2.set_title("Power Output Curves", fontsize=12)
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

print("\n🔑 Key Insights:")
print("   Maxon EC-i 40: Higher torque, lower speed → Heavy-duty applications")
print("   Faulhaber 2264: Lower torque, higher speed → High-speed applications")
print(
  f"   Peak power Maxon: {np.max(power_large):.0f}W vs Faulhaber: {np.max(power_small):.0f}W"
)
print("\n💡 Both motors automatically loaded from GitHub repository!")
print("   ✓ Cached locally for offline access")
print("   ✓ Real specifications from manufacturer datasheets")